In [ ]:
import cv2
import json
import os
import copy
import numpy as np

In [ ]:
def process_augmentation(video_path, json_path, video_output_dir, json_output_dir):
    """
    Applies 5 augmentations to a video and generates corresponding JSON metadata.
    Saves the videos and metadata to separate designated output directories.
    """
    # Ensure both output directories exist
    os.makedirs(video_output_dir, exist_ok=True)
    os.makedirs(json_output_dir, exist_ok=True)

    # 1. Load original metadata
    if not os.path.exists(json_path):
        print(f"Error: JSON file {json_path} not found.")
        return

    with open(json_path, 'r') as f:
        original_meta = json.load(f)

    video_id = original_meta.get("video_id", "unknown")

    # 2. Setup VideoCapture
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error: Could not open video {video_path}")
        return

    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')

    # Define augmentation suffixes
    augmentations = ['flip', 'bright', 'rotate', 'slow', 'zoom']

    # 3. Setup VideoWriters to save in the video output directory
    writers = {}
    for aug in augmentations:
        out_name = os.path.join(video_output_dir, f"{video_id}_{aug}.mp4")
        writers[aug] = cv2.VideoWriter(out_name, fourcc, fps, (width, height))

    # Rotation matrix setup (10 degrees)
    center = (width // 2, height // 2)
    rot_matrix = cv2.getRotationMatrix2D(center, 10, 1.0)

    # Zoom setup (Crop 10% from each side dynamically)
    crop_w = int(width * 0.1)
    crop_h = int(height * 0.1)

    print(f"Processing augmentations for {video_id}...")

    # 4. Process frames
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # A. Flip (Horizontal)
        flip = cv2.flip(frame, 1)
        writers['flip'].write(flip)

        # B. Brightness
        bright = cv2.convertScaleAbs(frame, alpha=1.2, beta=30)
        writers['bright'].write(bright)

        # C. Rotate
        rotate = cv2.warpAffine(frame, rot_matrix, (width, height))
        writers['rotate'].write(rotate)

        # D. Slow motion (write same frame twice, keeps same FPS)
        writers['slow'].write(frame)
        writers['slow'].write(frame)

        # E. Zoom (dynamic crop)
        crop = frame[crop_h:height-crop_h, crop_w:width-crop_w]
        zoom = cv2.resize(crop, (width, height))
        writers['zoom'].write(zoom)

    # Release resources
    cap.release()
    for w in writers.values():
        w.release()

    # 5. Generate and save new JSON metadata to the JSON output directory
    for aug in augmentations:
        # Deep copy to avoid modifying the original dict
        new_meta = copy.deepcopy(original_meta)

        # Update the video ID to match the new file
        new_meta["video_id"] = f"{video_id}_{aug}"

        # Adjust start/end frames ONLY for slow motion
        if aug == 'slow':
            new_meta["frame_start"] = new_meta.get("frame_start", 1) * 2
            new_meta["frame_end"] = new_meta.get("frame_end", 2) * 2

        # Save the new JSON file
        out_json_path = os.path.join(json_output_dir, f"{new_meta['video_id']}.json")
        with open(out_json_path, 'w') as f:
            json.dump(new_meta, f, indent=2)

    print(f"Successfully generated 5 augmented videos in '{video_output_dir}' and JSONs in '{json_output_dir}' for {video_id}")

In [ ]:
if __name__ == "__main__":
    # Target directories on Google Drive (used for both input and output)
    video_dir = "/content/drive/MyDrive/wlasl_train_data"
    json_dir = "/content/drive/MyDrive/instance_metadata"

    # Suffixes to check so we don't accidentally augment already-augmented videos
    aug_suffixes = ['_flip', '_bright', '_rotate', '_slow', '_zoom']

    if not os.path.exists(video_dir):
        print(f"Directory not found: {video_dir}")
    else:
        # Grab a snapshot of all mp4 files currently in the directory
        video_files = [f for f in os.listdir(video_dir) if f.endswith('.mp4')]

        print(f"Found {len(video_files)} total videos. Starting batch augmentation...")

        for filename in video_files:
            # Skip videos that are already augmented outputs
            if any(suffix in filename for suffix in aug_suffixes):
                continue

            video_id = filename.replace('.mp4', '')
            video_path = os.path.join(video_dir, filename)
            json_path = os.path.join(json_dir, f"{video_id}.json")

            # Check if the matching JSON exists before attempting augmentation
            if os.path.exists(json_path):
                process_augmentation(video_path, json_path, video_dir, json_dir)
            else:
                print(f"Warning: No matching JSON found for {filename}. Skipping.")

        print("Batch augmentation complete!")